# CME538 — Introduction to Data Science

# Tutorial 7 — Linear Regression, Feature Engineering & Generalization

## Prediction of Car Prices

In this tutorial, we will build a machine learning model to predict the price of a car using information about its characteristics.


By the end of the tutorial, we will be able to:

1. Explore relationships between numerical features and a target variable.
2. Interpret correlation without confusing it with causation.
3. Create useful engineered features.
4. Convert categorical variables into numerical features using one-hot encoding.
5. Fit a multiple linear regression model.
6. Evaluate predictions using MAE, RMSE, and $R^2$.
7. Distinguish training performance from performance on unseen data.
8. Explain why good training performance does not necessarily imply good generalization.

Our client wants to enter the Canadian automobile market and needs to understand the factors associated with car prices.

The client wants to know:

1. Which variables are useful for predicting car price?
2. How accurately can we predict the price?
3. How should we prepare the available variables before fitting a model?
4. How well does the model generalize to cars that were not used for training?


In [ ]:
# Import NumPy for numerical calculations and array operations.
# Import pandas for loading, cleaning, transforming, and analyzing tabular data.
# Import Matplotlib for creating plots.
# Import Seaborn for statistical visualizations.
# Import LinearRegression so that we can fit a linear regression model.
# Import train_test_split so that we can evaluate the model on unseen data.
# Import evaluation metrics that we will use to measure prediction performance.
# Set a consistent plotting style so that figures are easier to read.
# Set a random seed so that any random train-test split can be reproduced.


## 1. Read in and inspect the data

We will use the `CarPrices.csv` dataset. Each row represents one car, while the columns describe characteristics such as engine size, horsepower, fuel type, body type, and price.

Our **target variable** is `price`. The remaining variables are potential **features** for predicting price.

In [ ]:
# Load the car-price dataset from the CSV file into a pandas DataFrame.
# Display the first five rows so that we can see the structure of the dataset.


Before building a model, we should understand what kind of data we have.

In particular, we want to know:

* How many observations are there?
* What variables are available?
* Which variables are numerical?
* Which variables are categorical?
* Are there missing values?

This is an important part of the modeling process: **we should understand the data before choosing and fitting a model.**


In [ ]:
# Display the number of rows and columns in the dataset.
# Display the data type and non-null count for every column.
# Count the number of missing values in each column.


The dataset contains both **numerical** and **categorical** variables.

For example:

* `enginesize` is numerical.
* `horsepower` is numerical.
* `citympg` is numerical.
* `fueltype` is categorical.
* `carbody` is categorical.
* `drivewheel` is categorical.

A linear regression model requires numerical inputs, so later we will need to decide how to represent categorical variables.


## 2. Target distribution

Before fitting a model, examine the distribution of the target variable, `price`. The histogram shows where prices are concentrated and whether a small number of unusually expensive cars may affect model errors.

In [ ]:
# Create a figure with enough space for the target distribution.
# Plot the distribution of car prices.
# Add a descriptive title to the plot.
# Label the horizontal axis with the target variable and its units.
# Label the vertical axis with the number of observations.
# Display the completed figure.


Most cars are concentrated at lower prices, with fewer observations at substantially higher prices. This gives useful context when interpreting prediction errors.

## 3. Visualize a bivariate relationship

Before fitting a linear model, we should ask whether a linear relationship is even reasonable.

A scatter plot allows us to examine the relationship between one feature and the target.

Let's start with `enginesize`, since engine size is a plausible predictor of car price.

In [ ]:
# Create a scatter plot showing engine size against car price.
# Add a title describing the relationship being visualized.
# Label the horizontal axis with the predictor variable.
# Label the vertical axis with the target variable.
# Display the plot.


When looking at a bivariate relationship, we are interested in four things:

* **Form** — Is the relationship approximately linear?
* **Direction** — Does price tend to increase or decrease as engine size increases?
* **Strength** — How tightly do the observations follow the pattern?
* **Outliers** — Are there observations that behave very differently?

Here, we can see a generally positive relationship: cars with larger engines tend to have higher prices.

However, this does **not** mean that engine size causes price to increase. We are observing an association in the data.


## 4. Correlation

The Pearson correlation coefficient measures the strength and direction of a **linear association** between two numerical variables.

It ranges from -1 to 1:

* $r \approx 1$: strong positive linear association
* $r \approx -1$: strong negative linear association
* $r \approx 0$: weak linear association

A crucial warning:

> **Correlation does not imply causation.**

Correlation also only measures **linear** association. Two variables can have a strong nonlinear relationship while having a relatively small Pearson correlation.

In [ ]:
# Calculate the Pearson correlation between engine size and car price.


The correlation below summarizes the linear association between engine size and price. It is useful as a descriptive measure, but **correlation does not imply causation** and does not by itself select a final model.

We will use a small set of interpretable numerical variables—engine size, horsepower, curb weight, car width, and wheelbase—as inputs to a multiple regression model.

## 5. Feature Engineering

**Feature engineering** means creating useful model inputs from existing information. We will create a numerical fuel-economy feature and extract a cleaner manufacturer feature from the car name.

### Feature 1 — Fuel Economy

The dataset contains both:

* `citympg`
* `highwaympg`

These measure fuel economy under different driving conditions.

Instead of asking the model to separately learn from both variables, we can construct a single weighted fuel-economy feature.

The weights below are a modeling choice. They are not a physical law; they simply provide an example of domain-informed feature engineering.


In [ ]:
# Combine city and highway fuel economy into one engineered feature.
# Display the new feature alongside the original fuel-economy variables.


### Feature 2 — Manufacturer

The original `CarName` column contains both the manufacturer and the specific vehicle model.

For example:

`alfa-romero giulia`

contains:

* manufacturer: `alfa-romero`
* model: `giulia`

The specific model names create many categories with very few observations, so we will extract the manufacturer name.

This is a useful example of transforming an existing variable into a representation that is more appropriate for modeling.


In [ ]:
# Extract the first word from each car name as an initial manufacturer label.
# Display the original car name and extracted manufacturer side by side.


The dataset contains several spelling inconsistencies in manufacturer names.

For example, some observations use:

* `maxda` instead of `mazda`
* `porcshce` instead of `porsche`
* `toyouta` instead of `toyota`
* `vokswagen` or `vw` instead of `volkswagen`

If we leave these errors unchanged, the model will treat them as different manufacturers.


In [ ]:
# Define a dictionary that maps incorrect manufacturer labels to their corrected labels.
# Replace incorrect manufacturer labels using the correction dictionary.
# Display the unique manufacturer names after cleaning.


## 6. One-hot encoding a categorical feature

Linear regression requires numerical inputs. We will encode **one** categorical variable, `carbody`, to demonstrate one-hot encoding. Other categorical variables are intentionally left out here so that the example stays focused.

One-hot encoding creates a separate 0/1 feature for each category. We will drop one category to avoid redundant information in the linear model.

In [ ]:
# Display the distinct car-body categories before encoding.


In [ ]:
# Convert car-body categories into one-hot encoded numerical columns.
# Display the first five rows to verify that the categorical variable has been encoded.


The original `carbody` column is now represented by numerical indicator columns. With `drop_first=True`, one body type is the reference category and each remaining indicator records whether a car belongs to that body type.

## 7. Build the regression dataset

Our target is `price`. The model will use five numerical vehicle characteristics, the engineered fuel-economy feature, and the one-hot encoded car-body indicators. This compact feature set lets us focus on the modeling workflow.

In [ ]:
# Define the numerical and engineered features used for prediction.
# Find the one-hot encoded car-body columns and add them to the feature list.
# Create the feature matrix X and target vector y.
# Verify the model inputs.


## 8. Train-test split

To assess performance on a new car, split the observations into **training data** for fitting the model and **test data** held back for evaluation. Test performance estimates how well the model generalizes to unseen observations.

In [ ]:
# Split the observations into training and test sets.
# Display the number of observations in the training set.
# Display the number of observations in the test set.


## 9. Fit a multiple linear regression model

With multiple features, our linear regression model has the form

$$
\hat{y}
=

\theta_0
+
\theta_1x_1
+
\theta_2x_2
+
\cdots
+
\theta_px_p.
$$

Each feature has its own coefficient.

The model learns these coefficients from the training data by minimizing squared error.

We will use scikit-learn's `LinearRegression` estimator.

In [ ]:
# Create a linear regression model with an intercept term.
# Fit the model using only the training observations.
# Generate price predictions for the training observations.
# Generate price predictions for the previously unseen test observations.


## 10. Evaluate the model

We will use three common regression metrics:

### MAE — Mean Absolute Error

MAE measures the average absolute difference between the true and predicted values.

It is expressed in the same units as the target.

For car prices, an MAE of $2,000 means that predictions are off by about $2,000 on average.

### RMSE — Root Mean Squared Error

RMSE also has the same units as the target, but it penalizes large errors more strongly because the errors are squared before taking the square root.

### $R^2$

$R^2$ measures the proportion of variation in the target that is explained by the fitted model relative to a baseline that always predicts the mean.

Higher $R^2$ is generally better, although it should not be interpreted in isolation.

In [ ]:
# Calculate the mean absolute error on the training data.
# Calculate the mean absolute error on the test data.
# Calculate the root mean squared error on the training data.
# Calculate the root mean squared error on the test data.
# Calculate R-squared on the training data.
# Calculate R-squared on the test data.
# Print all evaluation metrics so that we can compare training and test performance.


### Training versus test performance

Compare performance on the training and test sets. If training errors are much lower than test errors, the model may be overfitting. Similar training and test performance is evidence that the model is generalizing more successfully.

In [ ]:
# Create a small table comparing training and test performance.
# Display the comparison table.


### Actual versus predicted test prices

Each point below represents a test-set car. Points close to the diagonal line have predicted prices close to their actual prices; larger vertical gaps indicate larger prediction errors.

In [ ]:
# Plot actual test prices against the corresponding predictions.
# Add the y = x reference line for perfect predictions.
# Add a descriptive title to the visualization.
# Label the horizontal axis with the observed prices.
# Label the vertical axis with the model predictions.
# Display the legend so that the reference line is identified.
# Display the completed figure.
